In [33]:
!pip install torch-geometric -q

In [34]:
import os, pickle, itertools, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_max_pool
from tqdm.notebook import tqdm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

In [35]:
from google.colab import drive
drive.mount('/content/drive', )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [36]:
BASE_PATH  = '/content/drive/MyDrive/team3_xai_gnn'
DATA_DIR   = os.path.join(BASE_PATH, 'preprocessed')
OUTPUT_DIR = os.path.join(BASE_PATH, 'gineconv_results')
INNER_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'inner_results.pkl')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [37]:
class GINEConvNet(nn.Module):
    def __init__(self, node_in, edge_in, hidden, dropout):
        super().__init__()

        # input projection
        self.input_proj = nn.Linear(node_in, hidden)
        self.input_norm = nn.BatchNorm1d(hidden)

        # GINE needs edge_attr to have the same dimension as node features
        self.edge_encoder1 = nn.Linear(edge_in, hidden)
        self.edge_encoder2 = nn.Linear(edge_in, hidden)

        # GINE layer 1
        nn1 = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.conv1 = GINEConv(nn1)
        self.norm1 = nn.BatchNorm1d(hidden)

        # GINE layer 2
        nn2 = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.conv2 = GINEConv(nn2)
        self.norm2 = nn.BatchNorm1d(hidden)

        self.dropout = nn.Dropout(dropout)

        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        # node projection
        x = self.input_proj(x)
        x = self.input_norm(x)
        x = x.relu()

        # edge projections for GINE
        edge_attr1 = self.edge_encoder1(edge_attr)
        edge_attr2 = self.edge_encoder2(edge_attr)

        # layer 1
        x = self.conv1(x, edge_index, edge_attr1)
        x = self.norm1(x)
        x = x.relu()
        x = self.dropout(x)

        # layer 2
        x = self.conv2(x, edge_index, edge_attr2)
        x = self.norm2(x)
        x = x.relu()
        x = self.dropout(x)

        # graph-level readout
        x_mean = global_mean_pool(x, batch)
        x_max  = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=-1)

        return self.classifier(x)

In [38]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        # --- step 1
        # standard cross entropy, one value per sample (no reduction yet)
        # this gives us -log(pt) for each sample
        ce_loss = nn.functional.cross_entropy(
            logits, targets,
            weight=self.weight,
            reduction='none'   # keep per-sample losses, don't average yet
        )

        # --- step 2
        # recover pt (the probability assigned to the correct class)
        # ce_loss = -log(pt)  →  pt = exp(-ce_loss)
        pt = torch.exp(-ce_loss)

        # --- step 3
        # apply the focal modulating factor (1 - pt)^gamma
        # when pt is high (easy example, model is confident) → factor is small → loss shrinks
        # when pt is low  (hard example, model is uncertain) → factor is ~1   → loss unchanged
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()

In [39]:
# helper functions
def get_labels(dataset):
    return np.array([d.y.item() for d in dataset])


def compute_metrics(labels, preds, probs):
    return {
        'balanced_acc':  balanced_accuracy_score(labels, preds),
        'accuracy':      accuracy_score(labels, preds),
        'recall_mci':    recall_score(labels, preds, pos_label=1, zero_division=0),
        'recall_cn':     recall_score(labels, preds, pos_label=0, zero_division=0),
        'precision_mci': precision_score(labels, preds, pos_label=1, zero_division=0),
        'precision_cn':  precision_score(labels, preds, pos_label=0, zero_division=0),
        'f1_mci':        f1_score(labels, preds, pos_label=1, zero_division=0),
        'f1_cn':         f1_score(labels, preds, pos_label=0, zero_division=0),
        'f1_macro':      f1_score(labels, preds, average='macro', zero_division=0),
        'auc':           roc_auc_score(labels, probs[:, 1]) if len(np.unique(labels)) > 1 else float('nan'),
    }


def avg_metrics(metrics_list):
    keys = metrics_list[0].keys()
    return {k: float(np.mean([m[k] for m in metrics_list])) for k in keys}


def passes_filter(avg, thresholds):
    return all(avg.get(k, 0.0) >= v for k, v in thresholds.items())


def build_criterion(cfg, train_labels):
    # weights are [1.0, mci_weight]
    weight = torch.tensor([1.0, cfg['mci_weight']], dtype=torch.float).to(device)
    return FocalLoss(gamma=cfg['gamma'], weight=weight)

def build_model_and_opt(cfg):
    model = GINEConvNet(NODE_IN, EDGE_IN, HIDDEN, DROPOUT).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    return model, opt


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss   = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        all_labels.append(batch.y.cpu().numpy())
        all_preds.append(probs.argmax(axis=1))
        all_probs.append(probs)
    return (
        np.concatenate(all_labels),
        np.concatenate(all_preds),
        np.concatenate(all_probs, axis=0),
    )


def train_with_early_stopping(model, opt, criterion, train_loader, val_loader):
    best_bal_acc   = -1.0
    best_state     = None
    best_metrics   = None
    patience_count = 0

    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, criterion, opt)

        if not np.isfinite(loss):
            return model, {
                'balanced_acc': 0.0,
                'accuracy': 0.0,
                'recall_mci': 0.0,
                'recall_cn': 0.0,
                'precision_mci': 0.0,
                'precision_cn': 0.0,
                'f1_mci': 0.0,
                'f1_cn': 0.0,
                'f1_macro': 0.0,
                'auc': 0.0,
            }

        labels, preds, probs = evaluate(model, val_loader)

        if not np.isfinite(probs).all():
            return model, {
                'balanced_acc': 0.0,
                'accuracy': 0.0,
                'recall_mci': 0.0,
                'recall_cn': 0.0,
                'precision_mci': 0.0,
                'precision_cn': 0.0,
                'f1_mci': 0.0,
                'f1_cn': 0.0,
                'f1_macro': 0.0,
                'auc': 0.0,
            }

        metrics = compute_metrics(labels, preds, probs)

        if metrics['balanced_acc'] > best_bal_acc:
            best_bal_acc   = metrics['balanced_acc']
            best_state     = copy.deepcopy(model.state_dict())
            best_metrics   = metrics
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_metrics

In [40]:
# fixed params
NODE_IN  = 4
EDGE_IN  = 15
HIDDEN   = 32
DROPOUT  = 0.3
EPOCHS   = 50
PATIENCE = 15
BATCH    = 32
# k-folds
N_OUTER  = 5
N_INNER  = 3

# hyperparams for tuning
gammas        = [0.5, 0.75, 1.0]
mci_weights   = [1.25, 1.5, 1.75, 2.0]
lrs           = [1e-3, 3e-3]
weight_decays = [5e-4, 1e-3]

CONFIGS = [
    {'mci_weight': mw, 'gamma': g, 'lr': lr, 'wd': wd}
    for mw, g, lr, wd in itertools.product(mci_weights, gammas, lrs, weight_decays)
]

print(f'Total configs: {len(CONFIGS)}')
for i, c in enumerate(CONFIGS):
    print(f'  [{i:02d}] {c}')

Total configs: 48
  [00] {'mci_weight': 1.25, 'gamma': 0.5, 'lr': 0.001, 'wd': 0.0005}
  [01] {'mci_weight': 1.25, 'gamma': 0.5, 'lr': 0.001, 'wd': 0.001}
  [02] {'mci_weight': 1.25, 'gamma': 0.5, 'lr': 0.003, 'wd': 0.0005}
  [03] {'mci_weight': 1.25, 'gamma': 0.5, 'lr': 0.003, 'wd': 0.001}
  [04] {'mci_weight': 1.25, 'gamma': 0.75, 'lr': 0.001, 'wd': 0.0005}
  [05] {'mci_weight': 1.25, 'gamma': 0.75, 'lr': 0.001, 'wd': 0.001}
  [06] {'mci_weight': 1.25, 'gamma': 0.75, 'lr': 0.003, 'wd': 0.0005}
  [07] {'mci_weight': 1.25, 'gamma': 0.75, 'lr': 0.003, 'wd': 0.001}
  [08] {'mci_weight': 1.25, 'gamma': 1.0, 'lr': 0.001, 'wd': 0.0005}
  [09] {'mci_weight': 1.25, 'gamma': 1.0, 'lr': 0.001, 'wd': 0.001}
  [10] {'mci_weight': 1.25, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.0005}
  [11] {'mci_weight': 1.25, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.001}
  [12] {'mci_weight': 1.5, 'gamma': 0.5, 'lr': 0.001, 'wd': 0.0005}
  [13] {'mci_weight': 1.5, 'gamma': 0.5, 'lr': 0.001, 'wd': 0.001}
  [14] {'mci_weight':

In [41]:
# data loading
print('Loading data...')

outer_folds = []
for k in range(1, N_OUTER + 1):
    outer_folds.append({
        'train': torch.load(os.path.join(DATA_DIR, f'fold_{k}_train.pt'), weights_only=False),
        'val':   torch.load(os.path.join(DATA_DIR, f'fold_{k}_val.pt'), weights_only=False),
        'test':  torch.load(os.path.join(DATA_DIR, f'fold_{k}_test.pt'), weights_only=False),
    })

with open(os.path.join(DATA_DIR, 'inner_fold_ids.pkl'), 'rb') as f:
    inner_fold_ids = pickle.load(f)

print('Done.')
for k, fold in enumerate(outer_folds):
    print(f'  Outer fold {k+1}: train={len(fold["train"])}  val={len(fold["val"])}  test={len(fold["test"])}')

Loading data...
Done.
  Outer fold 1: train=431  val=77  test=127
  Outer fold 2: train=431  val=77  test=127
  Outer fold 3: train=431  val=77  test=127
  Outer fold 4: train=431  val=77  test=127
  Outer fold 5: train=431  val=77  test=127


In [42]:
# resume if training stopped midway
if os.path.exists(INNER_RESULTS_PATH):
    with open(INNER_RESULTS_PATH, 'rb') as f:
        inner_results = pickle.load(f)
    print(f'Resuming - {len(inner_results)} runs already complete.')
else:
    inner_results = {}

# creates inner split
def get_inner_split(outer_train, inner_fold_dict):
    inner_train = [g for g in outer_train if g.subject_id in inner_fold_dict['inner_train']]
    inner_val   = [g for g in outer_train if g.subject_id in inner_fold_dict['inner_val']]
    return inner_train, inner_val

# the following loop performs the hyperparamter tuning train/val phase
# it iterates over all outer and inner folds, training/validating all configurations
outer_bar = tqdm(range(N_OUTER), desc='Outer folds', position=0)
for outer_idx in outer_bar:
    outer_train = outer_folds[outer_idx]['train']

    inner_bar = tqdm(range(N_INNER), desc=f'  Inner folds', position=1, leave=False)
    for inner_idx in inner_bar:
        inner_train, inner_val = get_inner_split(outer_train, inner_fold_ids[outer_idx][inner_idx])
        train_labels = get_labels(inner_train)
        train_loader = DataLoader(inner_train, batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(inner_val,   batch_size=BATCH, shuffle=False)

        cfg_bar = tqdm(range(len(CONFIGS)), desc='    Configs', position=2, leave=False)
        for cfg_idx in cfg_bar:
            key = (outer_idx, inner_idx, cfg_idx)
            if key in inner_results:
                cfg_bar.set_postfix_str('skipped')
                continue

            cfg           = CONFIGS[cfg_idx]
            criterion     = build_criterion(cfg, train_labels)
            model, opt    = build_model_and_opt(cfg)
            _, metrics    = train_with_early_stopping(model, opt, criterion, train_loader, val_loader)

            inner_results[key] = metrics
            cfg_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                                recall_mci=f'{metrics["recall_mci"]:.3f}')

        # save after every inner fold
        with open(INNER_RESULTS_PATH, 'wb') as f:
            pickle.dump(inner_results, f)

print('Hyperparameter tuning comlpeted.')

# average metrics per config across all 15 runs
inner_avg = {}
for cfg_idx in range(len(CONFIGS)):
    runs = [inner_results[(o, i, cfg_idx)] for o in range(N_OUTER) for i in range(N_INNER)]
    inner_avg[cfg_idx] = avg_metrics(runs)

print('\nInner-loop averaged metrics per config:')
df_inner = pd.DataFrame([
    {'config': i, **inner_avg[i], **CONFIGS[i]}
    for i in range(len(CONFIGS))
]).set_index('config')
print(df_inner.to_string())
df_inner.to_csv(os.path.join(OUTPUT_DIR, 'inner_avg_metrics.csv'))

Resuming - 720 runs already complete.


Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Inner folds:   0%|          | 0/3 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

...

  Inner folds:   0%|          | 0/3 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

Hyperparameter tuning comlpeted.

Inner-loop averaged metrics per config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma     lr      wd
config                                                                                                                                                      
0           0.594882  0.690255    0.437869   0.751895       0.442300      0.761065  0.424912  0.754798  0.589855  0.638975        1.25   0.50  0.001  0.0005
1           0.550557  0.638999    0.402914   0.698201       0.399395      0.702160  0.391656  0.699239  0.545447  0.587505        1.25   0.50  0.001  0.0010
2           0.475388  0.523216    0.399070   0.551705       0.324037      0.596008  0.351993  0.571779  0.461886  0.501172        1.25   0.50  0.003  0.0005
3           0.547869  0.620011    0.425961   0.669776       0.377977      0.700380  0.390556  0.683399  0.536977  0.578732        1.25   0.50

In [46]:
INNER_FILTER = {
    'balanced_acc': 0.70,
    'recall_mci':   0.60,
    'recall_cn':    0.70,
}

surviving_configs = [
    i for i in range(len(CONFIGS))
    if passes_filter(inner_avg[i], INNER_FILTER)
]

print(f'Configs surviving inner filter: {len(surviving_configs)}/{len(CONFIGS)}')
for i in surviving_configs:
    print(f'  [{i:02d}] bal_acc={inner_avg[i]["balanced_acc"]:.3f}  '
          f'recall_mci={inner_avg[i]["recall_mci"]:.3f}  '
          f'recall_cn={inner_avg[i]["recall_cn"]:.3f}  |  {CONFIGS[i]}')

Configs surviving inner filter: 13/48
  [10] bal_acc=0.707  recall_mci=0.629  recall_cn=0.785  |  {'mci_weight': 1.25, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.0005}
  [11] bal_acc=0.702  recall_mci=0.602  recall_cn=0.802  |  {'mci_weight': 1.25, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.001}
  [16] bal_acc=0.712  recall_mci=0.602  recall_cn=0.822  |  {'mci_weight': 1.5, 'gamma': 0.75, 'lr': 0.001, 'wd': 0.0005}
  [22] bal_acc=0.711  recall_mci=0.634  recall_cn=0.789  |  {'mci_weight': 1.5, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.0005}
  [23] bal_acc=0.715  recall_mci=0.669  recall_cn=0.762  |  {'mci_weight': 1.5, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.001}
  [32] bal_acc=0.715  recall_mci=0.617  recall_cn=0.814  |  {'mci_weight': 1.75, 'gamma': 1.0, 'lr': 0.001, 'wd': 0.0005}
  [33] bal_acc=0.706  recall_mci=0.604  recall_cn=0.807  |  {'mci_weight': 1.75, 'gamma': 1.0, 'lr': 0.001, 'wd': 0.001}
  [34] bal_acc=0.713  recall_mci=0.636  recall_cn=0.790  |  {'mci_weight': 1.75, 'gamma': 1.0, 'lr': 0.003, 'wd':

In [47]:
# surviving configurations get trained on the full train set, and validated on the outer val set
CKPT_DIR          = os.path.join(OUTPUT_DIR, 'outer_checkpoints')
OUTER_VAL_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'outer_val_results.pkl')
os.makedirs(CKPT_DIR, exist_ok=True)

# resume if training stopped midway
if os.path.exists(OUTER_VAL_RESULTS_PATH):
    with open(OUTER_VAL_RESULTS_PATH, 'rb') as f:
        outer_val_results = pickle.load(f)
    print(f'Resuming — {len(outer_val_results)} runs already complete.')
else:
    outer_val_results = {}

# the following loop iterates over all outer folds, for all configurations
# each configuration gets trained on the whole train set and validated
# on the outer val set of each fold
cfg_bar = tqdm(surviving_configs, desc='Configs', position=0)
for cfg_idx in cfg_bar:
    cfg = CONFIGS[cfg_idx]

    outer_bar = tqdm(range(N_OUTER), desc=f'  Outer folds', position=1, leave=False)
    for outer_idx in outer_bar:
        key = (cfg_idx, outer_idx)
        if key in outer_val_results:
            outer_bar.set_postfix_str('skipped')
            continue

        outer_train  = outer_folds[outer_idx]['train']
        outer_val    = outer_folds[outer_idx]['val']
        train_labels = get_labels(outer_train)
        train_loader = DataLoader(outer_train, batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(outer_val,   batch_size=BATCH, shuffle=False)

        criterion    = build_criterion(cfg, train_labels)
        model, opt   = build_model_and_opt(cfg)
        model, metrics = train_with_early_stopping(model, opt, criterion, train_loader, val_loader)

        outer_val_results[key] = metrics
        outer_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                              recall_mci=f'{metrics["recall_mci"]:.3f}',
                              recall_cn=f'{metrics["recall_cn"]:.3f}')

        ckpt_path = os.path.join(CKPT_DIR, f'cfg{cfg_idx:02d}_outer{outer_idx+1}.pt')
        torch.save(model.state_dict(), ckpt_path)

        with open(OUTER_VAL_RESULTS_PATH, 'wb') as f:
            pickle.dump(outer_val_results, f)

print('Training and validation completed.')

# average outer val metrics per config across 5 folds
outer_val_avg = {
    i: avg_metrics([outer_val_results[(i, o)] for o in range(N_OUTER)])
    for i in surviving_configs
}

print('\nOuter-val averaged metrics per surviving config:')
df_val = pd.DataFrame([
    {'config': i, **outer_val_avg[i], **CONFIGS[i]}
    for i in surviving_configs
]).set_index('config')
print(df_val.to_string())
df_val.to_csv(os.path.join(OUTPUT_DIR, 'outer_val_avg_metrics.csv'))

Configs:   0%|          | 0/13 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

...

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

Training and validation completed.

Outer-val averaged metrics per surviving config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma     lr      wd
config                                                                                                                                                      
10          0.774870  0.846753    0.656667   0.893072       0.613458      0.914926  0.625577  0.903139  0.764358  0.797237        1.25   1.00  0.003  0.0005
11          0.762155  0.818182    0.670000   0.854310       0.532517      0.913603  0.592674  0.882825  0.737749  0.789197        1.25   1.00  0.003  0.0010
16          0.763068  0.818182    0.671667   0.854469       0.529775      0.915395  0.586804  0.883018  0.734911  0.790459        1.50   0.75  0.001  0.0005
22          0.757088  0.833766    0.630833   0.883342       0.578543      0.908084  0.596004  0.894778  0.745391  0.803525        

In [52]:
# filtering models based on performance on outer val
OUTER_VAL_FILTER = {
    'balanced_acc': 0.75,
    'recall_mci':   0.70,
    'recall_cn':    0.70,
}

final_configs = [
    i for i in surviving_configs
    if passes_filter(outer_val_avg[i], OUTER_VAL_FILTER)
]

print(f'Configs surviving outer val filter: {len(final_configs)}/{len(surviving_configs)}')
for i in final_configs:
    print(f'  [{i:02d}] bal_acc={outer_val_avg[i]["balanced_acc"]:.3f}  '
          f'recall_mci={outer_val_avg[i]["recall_mci"]:.3f}  '
          f'recall_cn={outer_val_avg[i]["recall_cn"]:.3f}  |  {CONFIGS[i]}')

if not final_configs:
    raise RuntimeError('No configs survived. Loosen OUTER_VAL_FILTER thresholds.')

Configs surviving outer val filter: 4/13
  [23] bal_acc=0.789  recall_mci=0.736  recall_cn=0.841  |  {'mci_weight': 1.5, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.001}
  [35] bal_acc=0.784  recall_mci=0.710  recall_cn=0.858  |  {'mci_weight': 1.75, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.001}
  [46] bal_acc=0.782  recall_mci=0.725  recall_cn=0.838  |  {'mci_weight': 2.0, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.0005}
  [47] bal_acc=0.770  recall_mci=0.709  recall_cn=0.832  |  {'mci_weight': 2.0, 'gamma': 1.0, 'lr': 0.003, 'wd': 0.001}


In [53]:
# testing
test_results = {i: {} for i in final_configs}

# the following loop tests all configurations on the test set of all 5 outer folds
cfg_bar = tqdm(final_configs, desc='Configs', position=0)
for cfg_idx in cfg_bar:
    cfg = CONFIGS[cfg_idx]

    outer_bar = tqdm(range(N_OUTER), desc='  Outer folds', position=1, leave=False)
    for outer_idx in outer_bar:
        ckpt_path = os.path.join(CKPT_DIR, f'cfg{cfg_idx:02d}_outer{outer_idx+1}.pt')
        model, _  = build_model_and_opt(cfg)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

        test_loader          = DataLoader(outer_folds[outer_idx]['test'], batch_size=BATCH, shuffle=False)
        labels, preds, probs = evaluate(model, test_loader)
        metrics              = compute_metrics(labels, preds, probs)

        test_results[cfg_idx][outer_idx] = {
            'metrics': metrics,
            'probs':   probs,
            'preds':   preds,
            'labels':  labels,
        }

        outer_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                              recall_mci=f'{metrics["recall_mci"]:.3f}',
                              recall_cn=f'{metrics["recall_cn"]:.3f}')

# per-config average across outer folds
test_avg = {
    i: avg_metrics([test_results[i][o]['metrics'] for o in range(N_OUTER)])
    for i in final_configs
}

print('\nTest averaged metrics per final config:')
df_test = pd.DataFrame([
    {'config': i, **test_avg[i], **CONFIGS[i]}
    for i in final_configs
]).set_index('config')
print(df_test.to_string())
df_test.to_csv(os.path.join(OUTPUT_DIR, 'test_avg_metrics.csv'))

Configs:   0%|          | 0/4 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]


Test averaged metrics per final config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma     lr      wd
config                                                                                                                                                      
23          0.672939  0.733858    0.574310   0.771567       0.383299      0.882218  0.456817  0.822118  0.639468  0.757912        1.50    1.0  0.003  0.0010
35          0.667405  0.757480    0.522824   0.811987       0.418555      0.877419  0.456452  0.841074  0.648763  0.739767        1.75    1.0  0.003  0.0010
46          0.696404  0.765354    0.588737   0.804072       0.453996      0.894532  0.490690  0.842953  0.666821  0.746792        2.00    1.0  0.003  0.0005
47          0.696415  0.743307    0.620658   0.772173       0.402305      0.894020  0.486910  0.828225  0.657567  0.763005        2.00    1.0  0.003  0.0010


In [54]:
# ensemble per outer fold
# current strat: weighted average of class probabilities,
# weight = outer val balanced accuracy of that model on that fold.

print('Ensemble results per outer fold:')
ensemble_results = {}

for outer_idx in range(N_OUTER):
    labels         = test_results[final_configs[0]][outer_idx]['labels']
    n              = len(labels)
    weighted_probs = np.zeros((n, 2))
    total_weight   = 0.0

    for cfg_idx in final_configs:
        weight          = outer_val_results[(cfg_idx, outer_idx)]['balanced_acc']
        weighted_probs += weight * test_results[cfg_idx][outer_idx]['probs']
        total_weight   += weight

    weighted_probs /= total_weight
    ensemble_preds  = weighted_probs.argmax(axis=1)
    metrics         = compute_metrics(labels, ensemble_preds, weighted_probs)

    ensemble_results[outer_idx] = {
        **metrics,
        'labels': labels,
        'preds':  ensemble_preds,
        'probs':  weighted_probs,
    }

    print(f'  Outer fold {outer_idx+1}: '
          f'bal_acc={metrics["balanced_acc"]:.3f}  '
          f'recall_mci={metrics["recall_mci"]:.3f}  '
          f'recall_cn={metrics["recall_cn"]:.3f}  '
          f'auc={metrics["auc"]:.3f}')

ensemble_avg = avg_metrics([
    {k: v for k, v in ensemble_results[o].items() if k not in ('labels', 'preds', 'probs')}
    for o in range(N_OUTER)
])
print(f'\nEnsemble average across folds:')
for k, v in ensemble_avg.items():
    print(f'  {k}: {v:.4f}')

df_ensemble = pd.DataFrame([
    {'outer_fold': o+1, **{k: v for k, v in ensemble_results[o].items() if k not in ('labels', 'preds', 'probs')}}
    for o in range(N_OUTER)
] + [{'outer_fold': 'avg', **ensemble_avg}])
df_ensemble.to_csv(os.path.join(OUTPUT_DIR, 'ensemble_results.csv'), index=False)
print(f'\nAll results saved to {OUTPUT_DIR}')

Ensemble results per outer fold:
  Outer fold 1: bal_acc=0.698  recall_mci=0.500  recall_cn=0.895  auc=0.760
  Outer fold 2: bal_acc=0.657  recall_mci=0.480  recall_cn=0.833  auc=0.725
  Outer fold 3: bal_acc=0.769  recall_mci=0.821  recall_cn=0.717  auc=0.838
  Outer fold 4: bal_acc=0.674  recall_mci=0.542  recall_cn=0.806  auc=0.729
  Outer fold 5: bal_acc=0.685  recall_mci=0.630  recall_cn=0.740  auc=0.768

Ensemble average across folds:
  balanced_acc: 0.6964
  accuracy: 0.7606
  recall_mci: 0.5945
  recall_cn: 0.7983
  precision_mci: 0.4308
  precision_cn: 0.8921
  f1_mci: 0.4937
  f1_cn: 0.8407
  f1_macro: 0.6672
  auc: 0.7638

All results saved to /content/drive/MyDrive/team3_xai_gnn/gineconv_results


In [55]:
for outer_idx in range(N_OUTER):
    print(f'=== Outer Fold {outer_idx+1} ===')

    for cfg_idx in final_configs:
        r  = test_results[cfg_idx][outer_idx]
        cm = confusion_matrix(r['labels'], r['preds'])
        c  = CONFIGS[cfg_idx]
        print(f'cfg{cfg_idx} (mw={c["mci_weight"]} g={c["gamma"]} lr={c["lr"]} wd={c["wd"]})')
        print(f'{"":10s}  Pred CN  Pred MCI')
        print(f'  True CN   {cm[0,0]:4d}     {cm[0,1]:4d}')
        print(f'  True MCI  {cm[1,0]:4d}     {cm[1,1]:4d}\n')

    er = ensemble_results[outer_idx]
    cm = confusion_matrix(er['labels'], er['preds'])
    print(f'ENSEMBLE')
    print(f'{"":10s}  Pred CN  Pred MCI')
    print(f'  True CN   {cm[0,0]:4d}     {cm[0,1]:4d}')
    print(f'  True MCI  {cm[1,0]:4d}     {cm[1,1]:4d}')
    print()

=== Outer Fold 1 ===
cfg23 (mw=1.5 g=1.0 lr=0.003 wd=0.001)
            Pred CN  Pred MCI
  True CN     88       17
  True MCI    13        9

cfg35 (mw=1.75 g=1.0 lr=0.003 wd=0.001)
            Pred CN  Pred MCI
  True CN     95       10
  True MCI    12       10

cfg46 (mw=2.0 g=1.0 lr=0.003 wd=0.0005)
            Pred CN  Pred MCI
  True CN     99        6
  True MCI    15        7

cfg47 (mw=2.0 g=1.0 lr=0.003 wd=0.001)
            Pred CN  Pred MCI
  True CN     79       26
  True MCI     9       13

ENSEMBLE
            Pred CN  Pred MCI
  True CN     94       11
  True MCI    11       11

=== Outer Fold 2 ===
cfg23 (mw=1.5 g=1.0 lr=0.003 wd=0.001)
            Pred CN  Pred MCI
  True CN     82       20
  True MCI    12       13

cfg35 (mw=1.75 g=1.0 lr=0.003 wd=0.001)
            Pred CN  Pred MCI
  True CN     80       22
  True MCI    14       11

cfg46 (mw=2.0 g=1.0 lr=0.003 wd=0.0005)
            Pred CN  Pred MCI
  True CN     90       12
  True MCI    13       12

cfg47 (m